<a href="https://colab.research.google.com/github/s30807/engineering_project/blob/main/engineering_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Biblioteki

In [1]:
import json
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch

Wczytanie pliku JSON z adnotacjami COCO

In [2]:
def load_coco_annotations(json_path):
    """
    Wczytuje adnotacje COCO z pliku JSON.

    Parameters
    ----------
    json_path : str
        Ścieżka do pliku JSON.

    Returns
    -------
    dict
        Słownik zawierający dane COCO.
    """

    with open(json_path, "r") as f:
        coco = json.load(f)

    return coco

Wczytanie zdjęcia

In [3]:
def load_image(image_path):
    """
    Wczytuje zdjęcie i konwertuje je z BGR do RGB.

    Parameters
    ----------
    image_path : str
        Ścieżka do zdjęcia.

    Returns
    -------
    np.ndarray
        Obraz w formacie RGB.
    """

    image = cv2.imread(image_path)

    if image is None:
        raise FileNotFoundError(
            f"Nie udało się wczytać obrazu: {image_path}"
        )

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    return image

Wczytanie adnotacji danego zdjecia

In [4]:
def load_annotations(coco,image_id):
  """
  Wczytuje adnotacje zdjęcia o podanym ID.

  Parameters
  ----------
  coco : dict
    Słownik zawierający dane COCO.

  image_id : int
    Identyfikator zdjecia.

  Returns
  -------
  list
    Lista adnotacji
  """

  annotations = []

  for annotation in coco["annotations"]:
    if annotation["image_id"] == image_id:
      annotations.append(annotation)

  return annotations

Tworzenie maski do zdjecia na podstawie adnotacji

In [5]:
def create_mask(image_info, annotations):
  """
  Tworzy maskę dla zdjęcia na podstawie adnotacji.

  Parameters
  ----------


  Returns
  -------
  np.ndarray
    Maska zdjęcia.
  """

  # Tlo skladajace sie z zer
  height = image_info["height"]
  width = image_info["width"]

  mask = np.zeros((height, width), dtype=np.uint8) # uint8 - int o wielkosci 8 bitow - 256 wartosci

  # Uzupelnienie maski kategoriami
  for annotation in annotations:

    category_id = annotation["category_id"]

    polygon = np.array(
        annotation["segmentation"][0],
        dtype=np.int32
    )

    polygon = polygon.reshape(-1, 2)

    cv2.fillPoly(
        mask,
        [polygon],
        category_id
    )

  return mask

Wizualizacja maski

In [6]:
def visualize_image_and_mask(image, mask):
    """
    Wyświetla zdjęcie oraz odpowiadającą mu maskę.
    """

    plt.figure(figsize=(12, 6))

    # Oryginalne zdjęcie
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Oryginalne zdjęcie")
    plt.axis("off")

    # Maska
    plt.subplot(1, 2, 2)
    plt.imshow(mask)
    plt.title("Maska")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

Mapowanie nazwy maski do jej id

In [7]:
def category_id_to_name():

  category_id_to_name = {}
  for category in coco["categories"]:
    category_id = category["id"]
    category_name = category["name"]

    category_id_to_name[category_id] = category_name

  category_id_to_name[0] = "background" # zmiana nazwy z Fasion na backgroubd

  return category_id_to_name

In [8]:
def categories_on_photo(mask):

  unique_categories = np.unique(mask)

  for category_id in unique_categories:
    category_name = category_id_to_name()[category_id]
    print(f"{category_id}: {category_name}")

In [9]:
def resize_image_and_mask(image, mask, size=(256, 256)):
    # Zmiana rozmiaru obrazu
    image = cv2.resize(
        image,
        size,
        interpolation=cv2.INTER_LINEAR
    )

    # Zmiana rozmiaru maski
    mask = cv2.resize(
        mask,
        size,
        interpolation=cv2.INTER_NEAREST
    )

    return image, mask

#MAIN

In [10]:
# # Static
# DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/INZ/dataset/train"
# JSON_PATH = "/content/drive/MyDrive/Colab Notebooks/INZ/dataset/_annotations.coco.json"

# # Wczytanie JSON
# coco = load_coco_annotations(JSON_PATH)

# # Wyswietlenie obrazu
# image_info = coco["images"][0]

# image_path = os.path.join(DATASET_PATH, image_info["file_name"])

# image = load_image(image_path)

# # Adnotacje do obrazu
# annotations = load_annotations(coco,image_info["id"])

# # Tworzenie maski dla obrazu
# mask = create_mask(image_info,annotations)

# # Wyswietlenie maski
# visualize_image_and_mask(image,mask)


In [11]:
# # ==========================================
# # Wyświetlenie wszystkich klas
# # ==========================================

# for category in coco["categories"]:
#     print(
#         f'id: {category["id"]}, '
#         f'name: {category["name"]}'
#     )

In [12]:
# print(category_id_to_name())

In [13]:
# categories_on_photo(mask)

In [14]:
# Klasa Dataset - łącząca wczytywanie w jedno miejsce (source: https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html#)

class ClothesDataset(Dataset):
  def __init__(self):
    self.dataset_path = "/content/drive/MyDrive/Colab Notebooks/INZ/dataset/train"
    self.json_path = "/content/drive/MyDrive/Colab Notebooks/INZ/dataset/_annotations.coco.json"

    self.coco = load_coco_annotations(self.json_path)

  def __len__(self):
    return len(self.coco["images"])

  def __getitem__(self,index):
    # Zdobywanie danych o obrazie
    image_info = self.coco["images"][index]
    image_path = os.path.join(self.dataset_path,image_info["file_name"])
    image = load_image(image_path)

    # Adnotacje do obrazu
    annotations = load_annotations(self.coco,image_info["id"])

    # Tworzenie maski dla obrazu
    mask = create_mask(image_info,annotations)

    # Resize obrazu i maski
    image, mask = resize_image_and_mask(
        image,
        mask
    )

    # Dane wyjsciowe
    image = torch.from_numpy(image) # zamieniamy na tensor

    # Normalizacja skaluje wartości wejściowe obrazu do bardziej odpowiedniego zakresu, co ułatwia i stabilizuje proces uczenia sieci neuronowej
    image = image.float()
    image = image / 255.0

    image = image.permute(2,0,1) # bo np.narray jest H W C, a tensor jest C H W

    # Maska nie jest normalizowana - musza zostac zachowane etykiety
    mask = torch.from_numpy(mask).long() # long do crossentropy

    return image, mask

In [15]:
# Sprawdzenie klasy dataset
dataset = ClothesDataset()

image, mask = dataset[0]

print("IMAGE")
print(type(image))
print(image.shape)
print(image.dtype)
print(image.min())
print(image.max())

print("\nMASK")
print(type(mask))
print(mask.shape)
print(mask.dtype)
print(mask.unique())

IMAGE
<class 'torch.Tensor'>
torch.Size([3, 256, 256])
torch.float32
tensor(0.)
tensor(1.)

MASK
<class 'torch.Tensor'>
torch.Size([256, 256])
torch.int64
tensor([ 0,  1,  4,  8,  9, 10])


In [16]:
# Test dataloader
dataset = ClothesDataset()

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

for images, masks in dataloader:
    print("Images:", images.shape)
    print("Masks:", masks.shape)
    break

Images: torch.Size([2, 3, 256, 256])
Masks: torch.Size([2, 256, 256])
